# Einstein Summation

$$\begin{equation}
\begin{split}
\left( A \cdot B \right)_{i,k} &= \sum_{j}\left( A_{i,j} \cdot B_{j,k} \right) \\
\left( A \cdot B \right)_{i,k} &= A_{i,j} \cdot B_{j,k}
\end{split}
\end{equation}$$

`einsum('ij,jk->ik')` # sum over j index


In [ ]:
import torch

a = torch.tensor([1,2,3]) # tensor with shape [3], so i ranges from 0 to 2
b = torch.tensor([4,5,6]) # tensor with shape [3], so j ranges from 0 to 2

# Outer product of a (3x1) and b (3x1)
# a⊗b = ab^T
# [[1 2 3]] [[4 5 6]   [4   5  6]
#            [4 5 6] = [8  10 12]
#            [4 5 6]]  [12 15 18]
# 31,13->33
outer_product = torch.outer(a,b)
outer_product_einsum = torch.einsum('i,j->ij', a, b)

attention_weights_manual = torch.empty((3,3))
for i in range(3):
  for j in range(3):
    attention_weights_manual[i,j] = a[i]*b[j]

print("Outer product")
print(outer_product)
print(outer_product_einsum)
print(attention_weights_manual)

Outer product
tensor([[ 4,  5,  6],
        [ 8, 10, 12],
        [12, 15, 18]])
tensor([[ 4,  5,  6],
        [ 8, 10, 12],
        [12, 15, 18]])
tensor([[ 4.,  5.,  6.],
        [ 8., 10., 12.],
        [12., 15., 18.]])


In [ ]:
import torch

# Dot product
# [1 2 3] [4] = 32
#         [5]
#         [6]
dot_product = torch.dot(a,b)
dot_product_einsum = torch.einsum('i,i->', a, b) # 'i,i->sums over the index i'

print("Dot product")
print(dot_product)
print(dot_product_einsum)

Dot product
tensor(32)
tensor(32)


In [ ]:
import torch

# Matmul
# [[1 2 3]  [[7  8]     [[58 64]
#  [4 5 6]]  [9 10]   =  [139 154]
#            [11 12]]
matrix_a = torch.tensor([[1,2,3],
                         [4,5,6]])
matrix_b = torch.tensor([[7,  8],
                         [9, 10],
                         [11,12]])
matmul = torch.matmul(matrix_a, matrix_b)
matmul_einsum = torch.einsum('ij,jk->ik', matrix_a, matrix_b) # $$\sum_j A_{ij} \cdot B_{jk}$$
matmul_einsum = torch.einsum('ik,kj->ij', matrix_a, matrix_b) # $$\sum_k A_{ik} \cdot B_{kj}$$ same thing as above
attention_weights_manual = torch.empty((2,2))
for i in range(2):
  for j in range(2):
    for key_h in range(3):
      attention_weights_manual[i,j] += matrix_a[i,key_h] * matrix_b[key_h,j]

print("Matrix multiplication")
print(matmul)
print(matmul_einsum)
print(attention_weights_manual)

Matrix multiplication
tensor([[ 58,  64],
        [139, 154]])
tensor([[ 58,  64],
        [139, 154]])
tensor([[ 58.,  64.],
        [139., 154.]])


In [ ]:
import torch

w = torch.tensor([[0, 1, 2],
                  [3, 4, 5]]) # 2x3
x = torch.tensor([[1, 1, 2]]) # 1x3

# Elementwise multiplication
# Einsum: row-column
# [[0 1 2]   [[1 1 2]    [[0 1  4]
#  [3 4 5]]   [1 1 2]] =  [3 4 10]]
elementwise = w * x # PyTorch broadcasts `x` to match the shape of `w`
elementwise_einsum = torch.einsum('ij,kj->ij', w, x)
print("Elementwise multiplication")
print(elementwise)
print(elementwise_einsum)

Elementwise multiplication
tensor([[ 0,  1,  4],
        [ 3,  4, 10]])
tensor([[ 0,  1,  4],
        [ 3,  4, 10]])


In [ ]:
import torch

# Matrix multiplication
# Einsum: row-row
# [[0 1 2]          = [[5]
#  [3 4 5]] * [[1]     [17]]
#              [1]
#              [2]]
# A_{i,j} \cdot B_{k,j}; summing over the index j
matmul = torch.matmul(w, x.transpose(0, 1))
matmul = w @ x.transpose(0, 1) # Equivalent to the line above
matmul_einsum = torch.einsum('ij,kj->ik', w, x) # Equivalent to the line above
print("Matrix multiplication")
print(matmul)
print(matmul_einsum)

Matrix multiplication
tensor([[ 5],
        [17]])
tensor([[ 5],
        [17]])


In [ ]:
import torch

a = torch.tensor([[1,2,3],
                  [4,5,6]])
b = torch.tensor([[7,8,9],
                  [10,11,12]])
torch.matmul(a,b.transpose(0,1)) # 2x3 * 3x2 = 2x2

tensor([[ 50,  68],
        [122, 167]])

In [ ]:
import torch

# Matrix multiplication
a = torch.tensor([[1,2,3,4],
                  [5,6,7,8],
                  [9,10,11,12]]) # 3x4
b = torch.tensor([[1,2,3,4,5],
                  [6,7,8,9,10],
                  [11,12,13,14,15],
                  [16,17,18,19,20]]) # 4x5
matmul = torch.matmul(a,b)
matmul_einsum = torch.einsum("ij,jk->ik", a, b)
attention_weights_manual = torch.empty((3,5))
for i in range(3):
  for key_h in range(5):
    for j in range(4):
      attention_weights_manual[i,key_h] += a[i,j] * b[j,key_h]
print(matmul)
print(matmul_einsum)
print(attention_weights_manual)

tensor([[110, 120, 130, 140, 150],
        [246, 272, 298, 324, 350],
        [382, 424, 466, 508, 550]])
tensor([[110, 120, 130, 140, 150],
        [246, 272, 298, 324, 350],
        [382, 424, 466, 508, 550]])
tensor([[110., 120., 130., 140., 150.],
        [246., 272., 298., 324., 350.],
        [382., 424., 466., 508., 550.]])


In [ ]:
import torch

# Batch matrix multiplication
a = torch.tensor([[[1,2,3,4],
                   [5,6,7,8],
                   [9,10,11,12]],
                  [[12,13,14,15],
                   [16,17,18,19],
                   [20,21,22,23]]]) # 2x3x4
b = torch.tensor([[[1,2,3,4,5],
                   [6,7,8,9,10],
                   [11,12,13,14,15],
                   [16,17,18,19,20]],
                  [[21,22,23,24,25],
                   [26,27,28,29,30],
                   [31,32,33,34,35],
                   [36,37,38,39,40]]]) # 2x4x5
matmul = torch.matmul(a, b)
matmul_einsum = torch.einsum("ijk,ikl->ijl", a, b)
attention_weights_manual = torch.zeros((2,3,5))
for i in range(2):
  for j in range(3):
    for l in range(5):
      for key_h in range(4):
        attention_weights_manual[i,j,l] += a[i,j,key_h] * b[i,key_h,l]
print(matmul)
print(matmul_einsum)
print(attention_weights_manual)

tensor([[[ 110,  120,  130,  140,  150],
         [ 246,  272,  298,  324,  350],
         [ 382,  424,  466,  508,  550]],

        [[1564, 1618, 1672, 1726, 1780],
         [2020, 2090, 2160, 2230, 2300],
         [2476, 2562, 2648, 2734, 2820]]])
tensor([[[ 110,  120,  130,  140,  150],
         [ 246,  272,  298,  324,  350],
         [ 382,  424,  466,  508,  550]],

        [[1564, 1618, 1672, 1726, 1780],
         [2020, 2090, 2160, 2230, 2300],
         [2476, 2562, 2648, 2734, 2820]]])
tensor([[[ 110.,  120.,  130.,  140.,  150.],
         [ 246.,  272.,  298.,  324.,  350.],
         [ 382.,  424.,  466.,  508.,  550.]],

        [[1564., 1618., 1672., 1726., 1780.],
         [2020., 2090., 2160., 2230., 2300.],
         [2476., 2562., 2648., 2734., 2820.]]])


In [ ]:
import torch

# Matrix Dialonal
x = torch.tensor([[1,2,3],
                  [4,5,6],
                  [7,8,9]])
diag = torch.diag(x)
diag_einsum = torch.einsum("ii->i", x)
attention_weights_manual = torch.empty(3)
for i in range(3):
  attention_weights_manual[i] = x[i][i]
print(diag)
print(diag_einsum)
print(attention_weights_manual)

tensor([1, 5, 9])
tensor([1, 5, 9])
tensor([1., 5., 9.])


In [ ]:
import torch

# Matrix Trace (sum of the elements of the diagonal of the input 2-D matrix)
x = torch.tensor([[1,2,3],
                  [4,5,6],
                  [7,8,9]])
trace = torch.trace(x)
trace_einsum = torch.einsum("ii->", x)
attention_weights_manual = torch.tensor(0)
for i in range(3):
  attention_weights_manual += x[i][i]
print(trace)
print(trace_einsum)
print(attention_weights_manual)

tensor(15)
tensor(15)
tensor(15)


# transpose

In [ ]:
import torch

a = torch.tensor([[1,2],
                  [3,4]]) # H, W
a_t = torch.transpose(a,0,1)
# gather the width dimension
print(f"a.shape: {a.shape}")
print(f"a:\n{a}")
print(f"a_t:\n{a_t}")
print(f"a_t[0]:\n{a_t[0]}")

a.shape: torch.Size([2, 2])
a:
tensor([[1, 2],
        [3, 4]])
a_t:
tensor([[1, 3],
        [2, 4]])
a_t[0]:
tensor([1, 3])


In [409]:
import torch

a = torch.tensor([[[0,1,2],
                   [3,4,5]],
                  [[6,7,8],
                   [9,10,11]]]) # N, H, W
a_t = torch.transpose(a, 0, 1) # H, N, W
# gather the height dimension
print(f"a.shape: {a.shape}")
print(f"a:\n{a}")
print(f"a_t.shape: {a_t.shape}")
print(f"a_t:\n{a_t}")
print(f"a_t[0][0]:\n{a_t[0][0]}")

a.shape: torch.Size([2, 2, 3])
a:
tensor([[[ 0,  1,  2],
         [ 3,  4,  5]],

        [[ 6,  7,  8],
         [ 9, 10, 11]]])
a_t.shape: torch.Size([2, 2, 3])
a_t:
tensor([[[ 0,  1,  2],
         [ 6,  7,  8]],

        [[ 3,  4,  5],
         [ 9, 10, 11]]])
a_t[0][0]:
tensor([0, 1, 2])


In [410]:
import torch

a = torch.tensor([[[0,1,2],
                   [3,4,5]],
                  [[6,7,8],
                   [9,10,11]]]) # N, H, W
a_t = torch.transpose(a, 1, 2) # N, W, H
# gather the width dimension
print(f"a.shape: {a.shape}")
print(f"a:\n{a}")
print(f"a_t.shape: {a_t.shape}")
print(f"a_t:\n{a_t}")
print(f"a_t[0]:\n{a_t[0][0]}")

a.shape: torch.Size([2, 2, 3])
a:
tensor([[[ 0,  1,  2],
         [ 3,  4,  5]],

        [[ 6,  7,  8],
         [ 9, 10, 11]]])
a_t.shape: torch.Size([2, 3, 2])
a_t:
tensor([[[ 0,  3],
         [ 1,  4],
         [ 2,  5]],

        [[ 6,  9],
         [ 7, 10],
         [ 8, 11]]])
a_t[0]:
tensor([0, 3])


# scaled dot product attention

In [ ]:
import torch

N = 2
query_len = key_len = 3
embed_dim = 4
query_h = torch.arange(N * query_len * embed_dim).view(N, query_len, embed_dim) # N, query_len, embed_dim
key_h = torch.arange(N * key_len * embed_dim).view(N, key_len, embed_dim) # N, key_len, embed_dim
key_t = torch.transpose(key_h, 1, 2) # N, embed_dim, key_len
# gather the embed_dim dimension
print(f"query:\n{query_h}")
print(f"key_t:\n{key_t}")
query_key_t = torch.bmm(query_h, key_t) # N, query_len, key_len
print(f"query_key_t:\n{query_key_t}")

query:
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
key_t:
tensor([[[ 0,  4,  8],
         [ 1,  5,  9],
         [ 2,  6, 10],
         [ 3,  7, 11]],

        [[12, 16, 20],
         [13, 17, 21],
         [14, 18, 22],
         [15, 19, 23]]])
query_key_t:
tensor([[[  14,   38,   62],
         [  38,  126,  214],
         [  62,  214,  366]],

        [[ 734,  950, 1166],
         [ 950, 1230, 1510],
         [1166, 1510, 1854]]])


# matmul vs einsum

In [ ]:
import torch

N = 2
query_len = key_len = 3
embed_dim = 4
num_heads = 2
head_dim = embed_dim // num_heads

query = torch.arange(N * query_len * embed_dim).view(N, query_len, num_heads, head_dim) # N, query_len, num_heads, head_dim
query_h = torch.transpose(query, 1, 2) # N, num_heads, query_len, head_dim

key = torch.arange(N * key_len * embed_dim).view(N, key_len, num_heads, head_dim) # N, key_len, num_heads, head_dim
key_h = torch.transpose(key, 1, 2) # N, num_heads, key_len, head_dim
key_t = torch.transpose(key_h, 2, 3) # N, num_heads, head_dim, key_len

query_key_t = torch.matmul(query_h, key_t) # N, num_heads, query_len, key_len
query_key_t_einsum = torch.einsum("nqhd,nkhd->nhqk", query, key)
# sum over the head_dim dimension

print(f"query:\n{query_h}")
print(f"key_t:\n{key_t}")
print(f"query_key_t:\n{query_key_t}")
print(f"query_key_t_einsum:\n{query_key_t_einsum}")

query:
tensor([[[[ 0,  1],
          [ 4,  5],
          [ 8,  9]],

         [[ 2,  3],
          [ 6,  7],
          [10, 11]]],


        [[[12, 13],
          [16, 17],
          [20, 21]],

         [[14, 15],
          [18, 19],
          [22, 23]]]])
key_t:
tensor([[[[ 0,  4,  8],
          [ 1,  5,  9]],

         [[ 2,  6, 10],
          [ 3,  7, 11]]],


        [[[12, 16, 20],
          [13, 17, 21]],

         [[14, 18, 22],
          [15, 19, 23]]]])
query_key_t:
tensor([[[[   1,    5,    9],
          [   5,   41,   77],
          [   9,   77,  145]],

         [[  13,   33,   53],
          [  33,   85,  137],
          [  53,  137,  221]]],


        [[[ 313,  413,  513],
          [ 413,  545,  677],
          [ 513,  677,  841]],

         [[ 421,  537,  653],
          [ 537,  685,  833],
          [ 653,  833, 1013]]]])
query_key_t_einsum:
tensor([[[[   1,    5,    9],
          [   5,   41,   77],
          [   9,   77,  145]],

         [[  13,   33,   53],
      

In [ ]:
import torch

N = 2
query_len = key_len = value_len = 3
embed_dim = 4
num_heads = 2
head_dim = embed_dim // num_heads

query = torch.arange(N * query_len * embed_dim).view(N, query_len, num_heads, head_dim) # N, query_len, num_heads, head_dim
query_h = torch.transpose(query, 1, 2) # N, num_heads, query_len, head_dim

key = torch.arange(N * key_len * embed_dim).view(N, key_len, num_heads, head_dim) # N, key_len, num_heads, head_dim
key_h = torch.transpose(key, 1, 2) # N, num_heads, key_len, head_dim
key_t = torch.transpose(key_h, 2, 3) # N, num_heads, head_dim, key_len

value = torch.arange(N * value_len * embed_dim).view(N, value_len, num_heads, head_dim) # N, value_len, num_heads, head_dim
value_h = torch.transpose(value, 1, 2) # N, num_heads, value_len, head_dim

query_key_t = torch.matmul(query_h, key_t) # N, num_heads, query_len, key_len

query_key_t_v = torch.matmul(query_key_t, value_h)
# N, num_heads, query_len, key_len * 
# N, num_heads, value_len, head_dim
# key_len == value_len, so the result is N, num_heads, query_len, head_dim
query_key_t_v = torch.permute(query_key_t_v, (0, 2, 1, 3))
# N, query_len, num_heads, head_dim

query_key_t_v_einsum = torch.einsum("nhql,nlhd->nqhd", query_key_t, value)

print(f"query_key_t:\n{query_key_t}")
print(f"value:\n{value_h}")
print(f"query_key_t_v.shape: {query_key_t_v.shape}")
print(f"query_key_t_v:\n{query_key_t_v}")
print(f"query_key_t_v_einsum.shape: {query_key_t_v_einsum.shape}")
print(f"query_key_t_v_einsum:\n{query_key_t_v_einsum}")

query_key_t:
tensor([[[[   1,    5,    9],
          [   5,   41,   77],
          [   9,   77,  145]],

         [[  13,   33,   53],
          [  33,   85,  137],
          [  53,  137,  221]]],


        [[[ 313,  413,  513],
          [ 413,  545,  677],
          [ 513,  677,  841]],

         [[ 421,  537,  653],
          [ 537,  685,  833],
          [ 653,  833, 1013]]]])
value:
tensor([[[[ 0,  1],
          [ 4,  5],
          [ 8,  9]],

         [[ 2,  3],
          [ 6,  7],
          [10, 11]]],


        [[[12, 13],
          [16, 17],
          [20, 21]],

         [[14, 15],
          [18, 19],
          [22, 23]]]])
query_key_t_v.shape: torch.Size([2, 3, 2, 2])
query_key_t_v:
tensor([[[[   92,   107],
          [  754,   853]],

         [[  780,   903],
          [ 1946,  2201]],

         [[ 1468,  1699],
          [ 3138,  3549]]],


        [[[20624, 21863],
          [29926, 31537]],

         [[27216, 28851],
          [38174, 40229]],

         [[33808, 35839],

# einsum vs for loop

In [505]:
import torch

N = 2
query_len = key_len = value_len = 3
embed_dim = 4
num_heads = 2
head_dim = embed_dim // num_heads

query_h = torch.arange(N * query_len * embed_dim).view(N, query_len, num_heads, head_dim) # N, query_len, num_heads, head_dim
key_h = torch.arange(N * key_len * embed_dim).view(N, key_len, num_heads, head_dim) # N, key_len, num_heads, head_dim
value_h = torch.arange(N * value_len * embed_dim).view(N, value_len, num_heads, head_dim) # N, value_len, num_heads, head_dim

query_key_t = torch.einsum("nqhd,nkhd->nhqk", query_h, key_h) # N, num_heads, query_len, key_len
# sum over the head_dim dimension
query_key_t_M = torch.zeros(N, num_heads, query_len, key_len, dtype=torch.long)
for n in range(N):
  for h in range(num_heads):
    for q in range(query_len):
      for k in range(key_len):
        # option 1: use dot product
        query_key_t_M[n,h,q,k] = torch.dot(query_h[n,q,h], key_h[n,k,h])
        # option 2: use element-wise multiplication
        # for d in range(head_dim):
        #   query_key_t_M[n,h,q,k] += query[n,q,h,d] * key[n,k,h,d]

print(f"query_key_t:\n{query_key_t}")
print(f"query_key_t_M:\n{query_key_t_M}")

query_key_t:
tensor([[[[   1,    5,    9],
          [   5,   41,   77],
          [   9,   77,  145]],

         [[  13,   33,   53],
          [  33,   85,  137],
          [  53,  137,  221]]],


        [[[ 313,  413,  513],
          [ 413,  545,  677],
          [ 513,  677,  841]],

         [[ 421,  537,  653],
          [ 537,  685,  833],
          [ 653,  833, 1013]]]])
query_key_t_M:
tensor([[[[   1,    5,    9],
          [   5,   41,   77],
          [   9,   77,  145]],

         [[  13,   33,   53],
          [  33,   85,  137],
          [  53,  137,  221]]],


        [[[ 313,  413,  513],
          [ 413,  545,  677],
          [ 513,  677,  841]],

         [[ 421,  537,  653],
          [ 537,  685,  833],
          [ 653,  833, 1013]]]])


In [506]:
query_key_t_v = torch.einsum("nhqk,nlhd->nqhd", query_key_t, value_h)
# sum over the key_len index and value_len index
query_key_t_v_einsum = torch.zeros(N, query_len, num_heads, head_dim, dtype=torch.long)
for n in range(N):
  for q in range(query_len):
    for h in range(num_heads):
      for d in range(head_dim):
        for k in range(key_len):
          for l in range(value_len):
            query_key_t_v_einsum[n,q,h,d] += query_key_t[n,h,q,k] * value_h[n,l,h,d]
print(f"query_key_t_v:\n{query_key_t_v}")
print(f"query_key_t_v_m:\n{query_key_t_v_einsum}")

query_key_t_v:
tensor([[[[   180,    225],
          [  1782,   2079]],

         [[  1476,   1845],
          [  4590,   5355]],

         [[  2772,   3465],
          [  7398,   8631]]],


        [[[ 59472,  63189],
          [ 86994,  91827]],

         [[ 78480,  83385],
          [110970, 117135]],

         [[ 97488, 103581],
          [134946, 142443]]]])
query_key_t_v_m:
tensor([[[[   180,    225],
          [  1782,   2079]],

         [[  1476,   1845],
          [  4590,   5355]],

         [[  2772,   3465],
          [  7398,   8631]]],


        [[[ 59472,  63189],
          [ 86994,  91827]],

         [[ 78480,  83385],
          [110970, 117135]],

         [[ 97488, 103581],
          [134946, 142443]]]])
